# DECORE: VGG16 Channel Pruning with Reinforcement Learning

Implementation of [DECORE](https://arxiv.org/abs/2106.06091) (Deep Compression with Reinforcement Learning) on CIFAR-10.

One agent per channel learns a single weight; a sigmoid + Bernoulli sample decides keep/drop. Agents are trained with REINFORCE using a reward that combines compression (channels dropped) and accuracy (penalty on wrong predictions).

## 1. Imports

In [15]:
import os
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision
import torchvision.transforms as transforms
from tqdm.auto import tqdm

# This notebook lives in notebooks/. Run from the repo root so relative paths
# (./data, ./*.pth checkpoints) resolve correctly. Idempotent across re-runs.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("working dir:", os.getcwd())

## 2. Configuration & hyperparameters

In [16]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# Fix seeds for reproducibility.
torch.manual_seed(42)

# --- Reproduction schedule (paper VGG16/CIFAR-10).
# The paper prunes an ALREADY-trained baseline, so we first train the full
# network for `baseline_epochs`, then run DECORE from that checkpoint.
# For a quick smoke test instead, set baseline_epochs=5, num_epochs=10,
# policy_training_stop_epoch=7.
batch_size = 256             # paper uses 256
baseline_epochs = 160        # pretrain full network to ~93-94% before pruning
num_epochs = 300             # DECORE joint training epochs
policy_training_stop_epoch = 260  # stop policy at 260, then fine-tune 40 epochs
learning_rate = 0.1          # baseline-training SGD LR (cosine annealing)
# decore_lr kept high on purpose: the net must stay PLASTIC to co-adapt to
# pruning. 0.01 was too gentle -> net memorized (100% train) and pruning stalled.
decore_lr = 0.1              # joint DECORE-phase SGD LR
agent_lr = 0.05              # Adam LR for policy agents (was 0.01); crosses p=0.5 faster

# --- Baseline caching: if False and a checkpoint exists, load it instead of
# retraining the full network (saves the ~160-epoch pretrain each run).
retrain_baseline = False
baseline_ckpt = 'vgg16_cifar10_baseline.pth'

# --- DECORE reward / REINFORCE hyperparameters.
# NOTE: R_acc_mean = train_acc - lambda*(1-train_acc) is only positive (i.e.,
# pruning is rewarded) when train_acc > lambda/(lambda+1). At ~99% train acc,
# lambda must be <~100 to prune; lambda=500 needs ~99.8% train acc.
lambda_penalty = 50          # DECORE-<lambda>; lower => more compression
print_every = 5              # print a summary line every N epochs (keeps output light)

# --- Stabilizers NOT described in the paper. Default OFF for a paper-literal
# run; set True for a lower-variance (but non-literal) REINFORCE.
use_reward_baseline = False  # subtract an EMA baseline from the reward
use_entropy_bonus = False    # add an entropy bonus to encourage exploration
entropy_coef = 0.01          # only used when use_entropy_bonus = True
baseline_momentum = 0.9      # only used when use_reward_baseline = True

Using device: mps


## 3. Data intake & preprocessing

CIFAR-10 is used at its native 32x32 resolution with CIFAR mean/std normalization. Training uses the standard random crop (padding 4) + horizontal flip augmentation.

In [17]:
cifar_mean = [0.4914, 0.4822, 0.4465]
cifar_std = [0.2470, 0.2435, 0.2616]

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar_mean, std=cifar_std),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar_mean, std=cifar_std),
])

In [18]:
# Cache CIFAR-10: only download if the extracted files are not already present.
data_root = os.path.abspath('./data')
already_cached = os.path.isdir(os.path.join(data_root, 'cifar-10-batches-py'))
need_download = not already_cached
print("CIFAR-10 cache found; skipping download." if already_cached
      else "CIFAR-10 not cached; will download (~170MB, one time).")

# num_workers=0 avoids macOS/Jupyter DataLoader worker-pipe leaks over long runs
# (the [Errno 35] crash). CIFAR at 32x32 is tiny, so this costs ~nothing in speed.
train_dataset = torchvision.datasets.CIFAR10(root=data_root, train=True, download=need_download, transform=transform_train)
train_loader = data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

test_dataset = torchvision.datasets.CIFAR10(root=data_root, train=False, download=need_download, transform=transform_test)
test_loader = data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

CIFAR-10 cache found; skipping download.


## 4. Model components

- **`Agent`**: one learnable weight per channel; `sigmoid(weight)` gives the keep-probability.
- **`PrunableConvBNReLU`**: a conv->BN->ReLU block that multiplies its output by a channel mask (the sampled keep/drop actions). Masking after BN+ReLU keeps the masked model equivalent to the physically pruned one.

In [19]:
class Agent(nn.Module):
    def __init__(self, num_channels):
        super(Agent, self).__init__()
        # DECORE: one learnable parameter (weight) per channel.
        # Init to 6.9 so sigmoid(6.9) ~= 0.99 (keep all channels initially).
        self.weights = nn.Parameter(torch.full((num_channels,), 6.9))

    def forward(self, state=None):
        # The state is vestigial in DECORE (always constant); the agent's own
        # weights ARE the per-channel importance scores.
        probs = torch.sigmoid(self.weights)
        return probs

In [20]:
class PrunableConvBNReLU(nn.Module):
    """conv -> BN -> ReLU, with the channel mask applied to the BLOCK output.

    Masking after BN+ReLU (rather than between conv and BN) makes the masked
    forward pass exactly equivalent to physically removing the channel: a
    zeroed block output contributes nothing to the next conv, just like a
    deleted channel would.
    """
    def __init__(self, conv_layer, bn_layer):
        super(PrunableConvBNReLU, self).__init__()
        self.conv = conv_layer
        self.bn = bn_layer
        self.relu = nn.ReLU(inplace=True)
        self.out_channels = conv_layer.out_channels
        self.channel_mask = torch.ones(self.out_channels).to(device)

    def forward(self, x):
        out = self.relu(self.bn(self.conv(x)))
        out = out * self.channel_mask.view(1, -1, 1, 1)
        return out

## 5. Model initialization

Build the CIFAR VGG-16-BN variant from scratch (each conv wrapped in a `PrunableConvBNReLU` block), then attach one `Agent` per block. The network is trained from random init (the paper prunes a trained baseline; here we train jointly).

In [21]:
# CIFAR VGG-16 variant (~14.98M params): 13 conv layers + a single Linear head.
# 32x32 input, five 2x2 max-pools -> 1x1 feature map -> Linear(512, 10).
cfg_vgg16 = [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M',
             512, 512, 512, 'M', 512, 512, 512, 'M']


class VGGCifar(nn.Module):
    def __init__(self, cfg, num_classes=10):
        super(VGGCifar, self).__init__()
        layers = []
        in_c = 3
        for v in cfg:
            if v == 'M':
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            else:
                conv = nn.Conv2d(in_c, v, kernel_size=3, padding=1, bias=False)
                bn = nn.BatchNorm2d(v)
                layers.append(PrunableConvBNReLU(conv, bn))
                in_c = v
        self.features = nn.Sequential(*layers)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(nn.Linear(512, num_classes))

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


vgg16 = VGGCifar(cfg_vgg16, num_classes=10).to(device)

In [22]:
conv_layer_indices = []
for idx, layer in enumerate(vgg16.features):
    if isinstance(layer, PrunableConvBNReLU):
        conv_layer_indices.append(idx)

agents = []
for idx in conv_layer_indices:
    num_channels = vgg16.features[idx].out_channels
    agent = Agent(num_channels).to(device)
    agents.append(agent)

def get_initial_state(num_channels):
    return torch.ones(num_channels).to(device)

## 6. Loss & optimizers

The network weights are optimized with SGD on the cross-entropy loss; the agents are optimized separately with Adam via REINFORCE.

In [23]:
criterion = nn.CrossEntropyLoss()

model_optimizer = optim.SGD(vgg16.parameters(), lr=decore_lr, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(model_optimizer, T_max=num_epochs)

agent_optimizers = [optim.Adam(agent.parameters(), lr=agent_lr) for agent in agents]

# EMA baseline per agent for REINFORCE variance reduction (Bug e).
reward_baselines = [0.0 for _ in agents]

## 7. Train / test / pruning-summary functions

In [24]:
def train(model, device, train_loader, optimizer, agents, agent_optimizers, epoch,
          lambda_penalty, baselines=None, phase_label=None):
    model.train()
    total_correct = 0
    total_samples = 0
    running_loss = 0.0
    phase = phase_label if phase_label is not None else ("policy" if agent_optimizers is not None else "finetune")

    for inputs, targets in train_loader:
        bs = inputs.size(0)
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        if agent_optimizers is not None:
            for agent_opt in agent_optimizers:
                agent_opt.zero_grad()

        log_probs_list = []
        entropies_list = []
        actions_list = []

        for agent, idx in zip(agents, conv_layer_indices):
            prunable_conv = model.features[idx]
            probs = agent()
            if agent_optimizers is not None:
                # Policy-learning phase: sample a stochastic Bernoulli mask
                # and record log-probs/entropy for the REINFORCE update.
                m = torch.distributions.Bernoulli(probs)
                actions = m.sample()
                log_probs_list.append(m.log_prob(actions))
                entropies_list.append(m.entropy())
                actions_list.append(actions)
                prunable_conv.channel_mask = actions.detach()
            else:
                # Fine-tuning phase: policies are frozen, so use the FIXED
                # deterministic mask (keep channels with prob >= 0.5). This
                # fine-tunes the actual pruned subnetwork, not random ones.
                actions = (probs >= 0.5).float()
                prunable_conv.channel_mask = actions.detach()

        outputs = model(inputs)
        classification_loss = criterion(outputs, targets)

        _, predicted = outputs.max(1)
        total_correct += predicted.eq(targets).sum().item()
        total_samples += bs
        running_loss += classification_loss.item() * bs

        classification_loss.backward()
        optimizer.step()

        if agent_optimizers is not None:
            R_acc_mean = torch.where(predicted == targets,
                                     torch.ones(bs, device=device),
                                     -lambda_penalty * torch.ones(bs, device=device)).mean()

            for i, (agent_opt, log_probs, entropy, actions) in enumerate(
                    zip(agent_optimizers, log_probs_list, entropies_list, actions_list)):
                R_iC = torch.sum(1 - actions)
                R_i = R_iC * R_acc_mean
                # Bug (e): subtract an EMA baseline to reduce gradient variance.
                if baselines is not None:
                    advantage = R_i - baselines[i]
                    baselines[i] = baseline_momentum * baselines[i] + (1 - baseline_momentum) * R_i.item()
                else:
                    advantage = R_i
                policy_loss = -log_probs.sum() * advantage
                if use_entropy_bonus:
                    policy_loss = policy_loss - entropy_coef * entropy.sum()
                policy_loss.backward()
                agent_opt.step()


    return running_loss / total_samples, 100. * total_correct / total_samples

In [25]:
def test(model, device, test_loader):
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            test_loss += criterion(outputs, targets).item() * targets.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    return 100. * correct / total, test_loss / total

In [26]:
def current_compression(agents, threshold=0.5):
    """Fraction (%) of channels currently below the keep threshold."""
    total = sum(a.weights.numel() for a in agents)
    dropped = 0
    for agent in agents:
        with torch.no_grad():
            dropped += (agent() < threshold).sum().item()
    return 100.0 * dropped / total


def calculate_pruning(model, agents, threshold=0.5):
    total_channels = 0
    total_pruned_channels = 0
    print("Per-layer pruning summary:")
    for agent, idx in zip(agents, conv_layer_indices):
        num_channels = model.features[idx].out_channels
        total_channels += num_channels
        with torch.no_grad():
            probs = agent()
        num_channels_to_keep = int((probs >= threshold).sum().item())
        num_channels_to_drop = num_channels - num_channels_to_keep
        total_pruned_channels += num_channels_to_drop
        bar = "#" * int(30 * num_channels_to_keep / num_channels)
        print(f"  layer {idx:2d}: keep {num_channels_to_keep:3d}/{num_channels:3d} "
              f"|{bar:<30}| drop {num_channels_to_drop:3d}")
    pruned_ratio = 100.0 * total_pruned_channels / total_channels
    print(f"Total: keep {total_channels - total_pruned_channels}/{total_channels} channels, "
          f"pruning ratio {pruned_ratio:.2f}%")
    return pruned_ratio

## 8. Baseline training (full network)

The paper prunes an **already-trained** network, so we first train the full VGG-16 (all channels kept) to convergence. Because the agents start at weight 6.9 (keep-prob ~0.99), calling `train` with `agent_optimizers=None` uses an all-ones mask and trains the complete network. The best baseline checkpoint is saved to `vgg16_cifar10_baseline.pth`.

In [27]:
if (not retrain_baseline) and os.path.exists(baseline_ckpt):
    # Reuse the cached baseline instead of retraining from scratch.
    vgg16.load_state_dict(torch.load(baseline_ckpt, map_location=device))
    baseline_best, _ = test(vgg16, device, test_loader)
    print(f"Loaded cached baseline '{baseline_ckpt}' (skipped training). Top-1: {baseline_best:.2f}%")
else:
    baseline_optimizer = optim.SGD(vgg16.parameters(), lr=learning_rate, momentum=0.9, weight_decay=5e-4)
    baseline_scheduler = optim.lr_scheduler.CosineAnnealingLR(baseline_optimizer, T_max=baseline_epochs)

    baseline_best = 0.0
    header = f"{'epoch':>7} | {'phase':^8} | {'lr':>7} | {'train acc':>9} | {'train loss':>10} | {'test acc':>8}"
    print(header)
    print("-" * len(header))

    for epoch in range(baseline_epochs):
        # agents are at init (keep-prob ~0.99) and agent_optimizers=None => full network.
        tr_loss, tr_acc = train(vgg16, device, train_loader, baseline_optimizer, agents,
                                None, epoch, lambda_penalty=0, phase_label="baseline")
        te_acc, te_loss = test(vgg16, device, test_loader)
        lr_now = baseline_optimizer.param_groups[0]["lr"]
        baseline_scheduler.step()

        marker = ""
        if te_acc > baseline_best:
            baseline_best = te_acc
            torch.save(vgg16.state_dict(), baseline_ckpt)
            marker = "  <- best"
        if (epoch + 1) % print_every == 0:
            print(f"{epoch + 1:>7} | {'baseline':^8} | {lr_now:>7.4f} | {tr_acc:>8.2f}% | "
                  f"{tr_loss:>10.3f} | {te_acc:>7.2f}%{marker}")

    print(f"\nBaseline top-1 accuracy: {baseline_best:.2f}%")

Loaded cached baseline 'vgg16_cifar10_baseline.pth' (skipped training). Top-1: 93.48%


## 9. DECORE joint training

Starting from the pretrained baseline, jointly train the network and the policies. After `policy_training_stop_epoch`, policy updates stop and the remaining epochs fine-tune the fixed pruned subnetwork. Best/final checkpoints are saved.

In [28]:
# Start DECORE from the best pretrained baseline checkpoint.
if os.path.exists(baseline_ckpt):
    vgg16.load_state_dict(torch.load(baseline_ckpt, map_location=device))
    print(f'Loaded pretrained baseline checkpoint: {baseline_ckpt}')

# Per-lambda checkpoint names so different DECORE runs don't overwrite each other.
best_ckpt = f'vgg16_cifar10_best_l{lambda_penalty}.pth'
final_ckpt = f'vgg16_cifar10_final_l{lambda_penalty}.pth'
agents_ckpt = f'agents_l{lambda_penalty}.pth'  # learned policies (which channels to prune)

best_accuracy = 0.0
history = []

header = f"{'epoch':>7} | {'phase':^8} | {'lr':>7} | {'train acc':>9} | {'train loss':>10} | {'test acc':>8} | {'compress':>8}"
print(header)
print("-" * len(header))

for epoch in range(num_epochs):
    in_policy_phase = epoch < policy_training_stop_epoch
    if in_policy_phase:
        tr_loss_acc = train(vgg16, device, train_loader, model_optimizer, agents,
                            agent_optimizers, epoch, lambda_penalty,
                            baselines=(reward_baselines if use_reward_baseline else None))
    else:
        tr_loss_acc = train(vgg16, device, train_loader, model_optimizer, agents,
                            None, epoch, lambda_penalty=0)
    tr_loss, tr_acc = tr_loss_acc
    te_acc, te_loss = test(vgg16, device, test_loader)
    comp = current_compression(agents)
    lr_now = model_optimizer.param_groups[0]["lr"]
    scheduler.step()

    marker = ""
    if te_acc > best_accuracy:
        best_accuracy = te_acc
        torch.save(vgg16.state_dict(), best_ckpt)
        torch.save([a.state_dict() for a in agents], agents_ckpt)
        marker = "  <- best"

    phase = "policy" if in_policy_phase else "finetune"
    if (epoch + 1) % print_every == 0:
        print(f"{epoch + 1:>7} | {phase:^8} | {lr_now:>7.4f} | {tr_acc:>8.2f}% | "
              f"{tr_loss:>10.3f} | {te_acc:>7.2f}% | {comp:>7.1f}%{marker}")
    history.append({"epoch": epoch + 1, "phase": phase, "train_acc": tr_acc,
                    "test_acc": te_acc, "compression": comp})

    # When policy learning ends, show the learned per-layer pruning plan.
    if epoch + 1 == policy_training_stop_epoch:
        print("\nPolicy training finished. Learned pruning plan:")
        calculate_pruning(vgg16, agents, threshold=0.5)
        print("Fine-tuning the fixed pruned subnetwork...\n")

torch.save(vgg16.state_dict(), final_ckpt)
print(f"\nBest test accuracy: {best_accuracy:.2f}%")

Loaded pretrained baseline checkpoint: vgg16_cifar10_baseline.pth
  epoch |  phase   |      lr | train acc | train loss | test acc | compress
---------------------------------------------------------------------------
      5 |  policy  |  0.1000 |    86.15% |      0.421 |   74.09% |    11.4%
     10 |  policy  |  0.0998 |    88.61% |      0.344 |   83.02% |    14.1%  <- best
     15 |  policy  |  0.0995 |    89.64% |      0.317 |   82.80% |    15.3%
     20 |  policy  |  0.0990 |    90.09% |      0.296 |   79.07% |    16.3%
     25 |  policy  |  0.0984 |    90.21% |      0.293 |   82.73% |    17.1%
     30 |  policy  |  0.0977 |    90.40% |      0.286 |   79.29% |    18.5%
     35 |  policy  |  0.0969 |    90.65% |      0.280 |   82.42% |    19.2%
     40 |  policy  |  0.0959 |    90.79% |      0.277 |   85.77% |    19.8%
     45 |  policy  |  0.0948 |    91.13% |      0.266 |   80.18% |    20.2%
     50 |  policy  |  0.0936 |    91.23% |      0.259 |   83.13% |    20.6%
     55 |  po

## 10. Physical pruning & compression metrics

Masking channels to zero does **not** make the model smaller or faster — the conv still computes every channel. To realize DECORE's compression we must *physically remove* the dropped channels and rebuild a smaller network:

- For each conv layer, keep only the output filters whose agent probability >= 0.5.
- The **next** conv layer must drop the corresponding **input** channels.
- The first classifier `Linear` must drop the flattened features of the removed final-conv channels.

We then count parameters and FLOPs (MACs) of the original vs. pruned model to report the paper's `Params(PR)` / `FLOPs(PR)` metrics.

In [29]:
import copy


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def count_flops(model, input_size=(1, 3, 32, 32)):
    """Count multiply-accumulate operations (MACs) for one forward pass."""
    flops = [0]

    def conv_hook(module, inp, out):
        oc, oh, ow = out.shape[1], out.shape[2], out.shape[3]
        kernel_ops = module.kernel_size[0] * module.kernel_size[1] * (module.in_channels // module.groups)
        flops[0] += oc * oh * ow * kernel_ops

    def linear_hook(module, inp, out):
        flops[0] += module.in_features * module.out_features

    hooks = []
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            hooks.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, nn.Linear):
            hooks.append(m.register_forward_hook(linear_hook))

    was_training = model.training
    model.eval()
    with torch.no_grad():
        model(torch.randn(input_size).to(next(model.parameters()).device))
    for h in hooks:
        h.remove()
    if was_training:
        model.train()
    return flops[0]

In [30]:
def get_keep_masks(agents, threshold=0.5):
    """Deterministic keep-mask per conv layer (True = keep the channel)."""
    masks = []
    for agent in agents:
        with torch.no_grad():
            probs = agent()
        keep = probs >= threshold
        if keep.sum() == 0:  # safety: never fully delete a layer
            keep = torch.zeros_like(probs, dtype=torch.bool)
            keep[probs.argmax()] = True
        masks.append(keep)
    return masks


def build_pruned_vgg(model, agents, threshold=0.5, avgpool_spatial=1):
    """Rebuild the VGG with dropped channels physically removed.

    For each prunable block we slice the conv (out + in channels) and its BN;
    the next block drops the corresponding input channels; the final Linear
    drops the flattened features of the removed last-block channels.
    """
    keep_masks = get_keep_masks(agents, threshold)
    pruned = copy.deepcopy(model).to(device)

    new_features = []
    prev_keep_idx = torch.arange(3, device=device)  # RGB input channels
    block_ptr = 0
    for layer in model.features:
        if isinstance(layer, PrunableConvBNReLU):
            conv, bn = layer.conv, layer.bn
            keep = keep_masks[block_ptr].to(device)
            keep_idx = torch.nonzero(keep, as_tuple=False).squeeze(1)

            new_conv = nn.Conv2d(len(prev_keep_idx), len(keep_idx),
                                 kernel_size=conv.kernel_size, stride=conv.stride,
                                 padding=conv.padding, dilation=conv.dilation,
                                 groups=conv.groups, bias=conv.bias is not None)
            new_conv.weight.data.copy_(conv.weight.data[keep_idx][:, prev_keep_idx, :, :].clone())
            if conv.bias is not None:
                new_conv.bias.data.copy_(conv.bias.data[keep_idx].clone())

            new_bn = nn.BatchNorm2d(len(keep_idx))
            new_bn.weight.data.copy_(bn.weight.data[keep_idx].clone())
            new_bn.bias.data.copy_(bn.bias.data[keep_idx].clone())
            new_bn.running_mean.data.copy_(bn.running_mean.data[keep_idx].clone())
            new_bn.running_var.data.copy_(bn.running_var.data[keep_idx].clone())

            new_features.append(nn.Sequential(new_conv, new_bn, nn.ReLU(inplace=True)).to(device))
            prev_keep_idx = keep_idx
            block_ptr += 1
        else:
            new_features.append(copy.deepcopy(layer))
    pruned.features = nn.Sequential(*new_features).to(device)

    # --- Rebuild the classifier Linear to match new flattened dim ---
    last_keep_idx = prev_keep_idx
    spatial = avgpool_spatial * avgpool_spatial
    old_fc = model.classifier[0]
    in_idx = (last_keep_idx.view(-1, 1) * spatial +
              torch.arange(spatial, device=device)).view(-1)
    new_fc = nn.Linear(len(last_keep_idx) * spatial, old_fc.out_features)
    new_fc.weight.data.copy_(old_fc.weight.data[:, in_idx].clone())
    new_fc.bias.data.copy_(old_fc.bias.data.clone())
    pruned.classifier = nn.Sequential(new_fc.to(device)).to(device)

    return pruned

In [31]:
input_size = (1, 3, 32, 32)

# Build the compressed model from the best fine-tuned checkpoint.
best_ckpt = f'vgg16_cifar10_best_l{lambda_penalty}.pth'
agents_ckpt = f'agents_l{lambda_penalty}.pth'
if os.path.exists(best_ckpt):
    vgg16.load_state_dict(torch.load(best_ckpt, map_location=device))
# Restore learned policies from disk if present (crash / fresh-kernel recovery).
if os.path.exists(agents_ckpt):
    for a, s in zip(agents, torch.load(agents_ckpt, map_location=device)):
        a.load_state_dict(s)

orig_params = count_parameters(vgg16)
orig_flops = count_flops(vgg16, input_size)

pruned_model = build_pruned_vgg(vgg16, agents, threshold=0.5, avgpool_spatial=1)

pruned_params = count_parameters(pruned_model)
pruned_flops = count_flops(pruned_model, input_size)

pruned_acc, _ = test(pruned_model, device, test_loader)
params_pr = 100 * (1 - pruned_params / orig_params)
flops_pr = 100 * (1 - pruned_flops / orig_flops)

line = "=" * 52
print(line)
print(f"  DECORE-{lambda_penalty} compression report (VGG16 / CIFAR-10)")
print(line)
print(f"  {'metric':<12}{'baseline':>12}{'pruned':>12}{'reduction':>13}")
print("-" * 52)
print(f"  {'Params':<12}{orig_params/1e6:>10.2f}M{pruned_params/1e6:>11.2f}M{params_pr:>12.1f}%")
print(f"  {'FLOPs':<12}{orig_flops/1e6:>10.2f}M{pruned_flops/1e6:>11.2f}M{flops_pr:>12.1f}%")
print("-" * 52)
print(f"  {'Top-1 acc':<12}{best_accuracy:>11.2f}%{pruned_acc:>11.2f}%")
print(line)

pruned_ckpt = f'vgg16_cifar10_pruned_l{lambda_penalty}.pth'
torch.save(pruned_model, pruned_ckpt)
print(f"Saved compressed model -> {pruned_ckpt}")

  DECORE-50 compression report (VGG16 / CIFAR-10)
  metric          baseline      pruned    reduction
----------------------------------------------------
  Params           14.72M       5.43M        63.1%
  FLOPs           313.20M     192.03M        38.7%
----------------------------------------------------
  Top-1 acc         93.76%      93.76%
Saved compressed model -> vgg16_cifar10_pruned_l50.pth
